# Random Forest Regressor — After Feature Engineering

**Author:** Liz Choi  
**Fix applied:** Removed `PPSF` (Price Per Square Foot) from the feature list.

### Why PPSF caused data leakage
`PPSF = ClosePrice / LivingArea`

Since `ClosePrice` is the **target variable**, including `PPSF` as a feature directly exposes the answer to the model during training. This resulted in unrealistically perfect metrics (Train R² = 0.9991, Test MAPE = 0.63%) that do **not** reflect genuine predictive ability.

**Corrected feature set removes PPSF**, retaining only features that would be known *before* a sale closes.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error

## 2. Load Feature-Engineered Data

In [2]:
train = pd.read_csv('train_cleaned_fe.csv')
test  = pd.read_csv('test_cleaned_fe.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

Train shape: (67727, 31)
Test shape:  (10324, 31)


## 3. Log-Transform Target Variable

In [3]:
train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice']  = np.log(test['ClosePrice'])

## 4. Select Features

**Important:** `PPSF` is intentionally excluded.

| Feature | Reason |
|---|---|
| ~~PPSF~~ | ❌ Removed — equals ClosePrice/LivingArea (data leakage) |
| LivingArea | ✅ Known before sale |
| BathroomsTotalInteger | ✅ Known before sale |
| BedroomsTotal | ✅ Known before sale |
| Bed_Bath_Ratio | ✅ Engineered from known features |
| Property_Age | ✅ Derived from YearBuilt |
| Months_From_Dec_2025 | ✅ Temporal feature |
| Sqft_Per_Bedroom | ✅ Engineered from known features |
| Lot_Utilization | ✅ Engineered from known features |

#### ZIP Code Median Price Encoding
Location plays a critical role in real estate pricing. Instead of using ZIP codes directly as categorical variables, we encode them using a median price signal.

Steps performed:

Calculate the median log closing price for each ZIP code in the training dataset.
Map this value to each property based on its ZIP code.
Assign the value to a new feature called ZipMedianPrice.
This approach captures neighborhood-level pricing patterns while avoiding the high dimensionality of one-hot encoding.

If a ZIP code appears in the test set but not in the training set, it is assigned the global median log price to prevent missing values.

In [5]:
#ZIP median encoding
zip_median_price = train.groupby('PostalCode')['LogClosePrice'].median()

train['ZipMedianPrice'] = train['PostalCode'].map(zip_median_price)
test['ZipMedianPrice']  = test['PostalCode'].map(zip_median_price)

global_median = train['LogClosePrice'].median()
test['ZipMedianPrice'] = test['ZipMedianPrice'].fillna(global_median)

In [6]:
rf_features = [
    'LivingArea',
    'BathroomsTotalInteger',
    'BedroomsTotal',
    # 'PPSF' -- REMOVED: PPSF = ClosePrice / LivingArea → data leakage
    'Bed_Bath_Ratio',
    'Property_Age',
    'Months_From_Dec_2025',
    'Sqft_Per_Bedroom',
    'Lot_Utilization',
    'ZipMedianPrice'
]

for col in rf_features:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col],  errors='coerce')

## 5. Handle Missing Values

In [7]:
for df in [train, test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

train = train.dropna(subset=rf_features + ['LogClosePrice'])
test  = test.dropna(subset=rf_features  + ['LogClosePrice'])

print(f'Train after dropna: {train.shape}')
print(f'Test  after dropna: {test.shape}')

Train after dropna: (67669, 33)
Test  after dropna: (10314, 33)


## 6. Prepare Feature Matrices

In [8]:
X_train = train[rf_features].astype(np.float32)
X_test  = test[rf_features].astype(np.float32)

y_train = train['LogClosePrice']
y_test  = test['LogClosePrice']

## 7. Train Random Forest Regressor

Same hyperparameters as the original notebook:
- `n_estimators=600`, `max_depth=18`, `min_samples_leaf=20`

In [9]:
rf_model = RandomForestRegressor(
    n_estimators=600,
    max_depth=18,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print('Training complete.')

Training complete.


## 8. Generate Predictions

In [10]:
train_pred = rf_model.predict(X_train)
test_pred  = rf_model.predict(X_test)

## 9. Evaluate Model

- R² is reported on both log and actual scale
- MAPE and MdAPE are computed on the **actual ClosePrice scale** (after `np.exp()` back-transformation)

In [11]:
# R² on log scale
train_r2_log = r2_score(y_train, train_pred)
test_r2_log  = r2_score(y_test,  test_pred)

# Back-transform to actual dollar scale
y_train_d    = np.exp(y_train)
y_test_d     = np.exp(y_test)
train_pred_d = np.exp(train_pred)
test_pred_d  = np.exp(test_pred)

# R² on actual scale
train_r2_actual = r2_score(y_train_d, train_pred_d)
test_r2_actual  = r2_score(y_test_d,  test_pred_d)

# RMSE on actual scale
train_rmse = np.sqrt(mean_squared_error(y_train_d, train_pred_d))
test_rmse  = np.sqrt(mean_squared_error(y_test_d,  test_pred_d))

# MAPE on actual scale (corrected)
train_mape  = mean_absolute_percentage_error(y_train_d, train_pred_d) * 100
test_mape   = mean_absolute_percentage_error(y_test_d,  test_pred_d)  * 100

# MdAPE on actual scale (corrected)
train_mdape = np.median(np.abs((y_train_d - train_pred_d) / y_train_d)) * 100
test_mdape  = np.median(np.abs((y_test_d  - test_pred_d)  / y_test_d))  * 100

print('Random Forest Performance (PPSF removed — no leakage)')
print(f'Train R2 (log):    {train_r2_log:.4f} | Test R2 (log):    {test_r2_log:.4f}')
print(f'Train R2 (actual): {train_r2_actual:.4f} | Test R2 (actual): {test_r2_actual:.4f}')
print(f'Train RMSE ($):    {train_rmse:,.0f}  | Test RMSE ($):    {test_rmse:,.0f}')
print(f'Train MAPE:  {train_mape:.2f}%  | Test MAPE:  {test_mape:.2f}%')
print(f'Train MdAPE: {train_mdape:.2f}% | Test MdAPE: {test_mdape:.2f}%')

Random Forest Performance (PPSF removed — no leakage)
Train R2 (log):    0.9213 | Test R2 (log):    0.8866
Train R2 (actual): 0.8695 | Test R2 (actual): 0.8271
Train RMSE ($):    342,690  | Test RMSE ($):    387,558
Train MAPE:  12.46%  | Test MAPE:  14.94%
Train MdAPE: 8.80% | Test MdAPE: 10.36%


## 10. Results Summary Table

In [12]:
rf_results = pd.DataFrame({
    'Metric': [
        'R² (log scale)',
        'R² (actual scale)',
        'RMSE (actual $)',
        'MAPE (%) — actual scale',
        'MdAPE (%) — actual scale'
    ],
    'Training Set': [
        round(train_r2_log,    4),
        round(train_r2_actual, 4),
        round(train_rmse,      0),
        round(train_mape,      2),
        round(train_mdape,     2)
    ],
    'Testing Set': [
        round(test_r2_log,    4),
        round(test_r2_actual, 4),
        round(test_rmse,      0),
        round(test_mape,      2),
        round(test_mdape,     2)
    ]
})

rf_results

,Metric,Training Set,Testing Set
0,R² (log scale),0.9213,0.8866
1,R² (actual scale),0.8695,0.8271
2,RMSE (actual $),342690.0000,387558.0000
3,MAPE (%) — actual scale,12.4600,14.9400
4,MdAPE (%) — actual scale,8.8000,10.3600


## 11. Feature Importance

In [13]:
importances = pd.DataFrame({
    'Feature':    rf_features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

importances

,Feature,Importance
8,ZipMedianPrice,0.818370
0,LivingArea,0.153495
7,Lot_Utilization,0.008879
4,Property_Age,0.008203
6,Sqft_Per_Bedroom,0.003336
3,Bed_Bath_Ratio,0.002426
2,BedroomsTotal,0.002185
1,BathroomsTotalInteger,0.002130
5,Months_From_Dec_2025,0.000976


## 12. Interpretation

### Why the previous results were wrong
The original notebook included `PPSF = ClosePrice / LivingArea` as a feature. Since `ClosePrice` is the prediction target, the model was effectively given the answer during training. This produced unrealistically perfect results:
- Train R² (log) ≈ **0.9991** → now realistic ~0.90+
- Test MAPE ≈ **0.63%** → now realistic ~10-15%

### Corrected results
With `PPSF` removed, the model is evaluated honestly using only features that would be available *before* a home sale closes. The resulting MAPE and MdAPE reflect actual prediction accuracy on real home prices.

### Notes on metrics
- **R² (actual scale)** is lower than log scale R² — this is expected and normal
- **MdAPE < MAPE** indicates the model handles typical homes well but struggles more with extreme luxury prices
- Comparing to XGBoost and LightGBM on the same feature set provides a fair benchmark